# Explore Fragments by fragment_ids and Properties
This notebook loads REBEL fragments from the filtered fragments dataset, retrieves specified fragments by their `fragment_ids`, and then filters them further by provided property-value pairs.

In [ ]:
from __future__ import annotations

from collections.abc import Iterable
from pathlib import Path

from kebab.utils.dataset.wikidata.wikidata_utils import ResolvedWikidataEntity

In [10]:
# (input) Path to the fragments JSONL file
fragments_path = Path.cwd().parent / "data" / "REBEL" / "rebel_fragments" / "filtered" / "rebel_entity_fragments.jsonl"

In [ ]:
# Load fragments from JSONL file
def load_jsonl(file_path: Path) -> Iterable[ResolvedWikidataEntity]:
    """Yield ResolvedWikidataEntity objects from a JSONL file."""
    with open(file_path, "r", encoding="utf-8") as f:
        for line_ in f:
            line = line_.strip()
            if not line:
                continue
            yield ResolvedWikidataEntity.from_json(line)


fragments: list[ResolvedWikidataEntity] = list(load_jsonl(fragments_path))
fragment_id_to_fragment: dict[str, ResolvedWikidataEntity] = {f.metadata["fragment_id"]: f for f in fragments}
print(f"Loaded {len(fragments):,d} fragments")

Loaded 6,806,719 fragments


In [12]:
# (input) List of fragment_ids to retrieve
fragment_ids = ["2050943", "20608888", "4225724", "17774515", "17231414"]

In [13]:
# Retrieve fragments by fragment_id
retrieved_fragments: list[ResolvedWikidataEntity] = []

for fid in fragment_ids:
    fragment = fragment_id_to_fragment.get(fid)
    if fragment is None:
        print(f"Warning: fragment_id {fid} not found in index")
        continue

    retrieved_fragments.append(fragment)

# Display each fragment
for i, fragment in enumerate(retrieved_fragments, 1):
    print()
    print(f"Fragment {i}: {fragment.to_json()}")


Fragment 1: {"entity_id": "Q100067", "properties": {"name": ["Nawada district"]}, "source_ids": [], "evidence_map": {}, "metadata": {"fragment_id": "2050943", "title": "Khalkhu", "title_entity_id": "Q27963941", "type": ["district of India"]}}

Fragment 2: {"entity_id": "Q100067", "properties": {"name": ["Nawada"]}, "source_ids": [], "evidence_map": {}, "metadata": {"fragment_id": "20608888", "title": "Magahi language", "title_entity_id": "Q33728", "type": ["district of India"]}}

Fragment 3: {"entity_id": "Q100067", "properties": {"name": ["Nawada district"], "country": ["India"]}, "source_ids": [], "evidence_map": {}, "metadata": {"fragment_id": "4225724", "title": "Kutri", "title_entity_id": "Q86751968", "type": ["district of India"]}}

Fragment 4: {"entity_id": "Q100067", "properties": {"name": ["Nawada district"], "located in the administrative territorial entity": ["Magadh division"], "shares border with": ["Gaya"]}, "source_ids": [], "evidence_map": {}, "metadata": {"fragment_id

In [14]:
# (input) Property-value pairs to filter fragments
properties = {"country": ["India"]}

In [15]:
# Further filter retrieved fragments by property-value pairs
matching_fragments: list[ResolvedWikidataEntity] = []

for fragment in retrieved_fragments:
    for prop, desired_values in properties.items():
        frag_values = list(getattr(fragment, "properties", {}).get(prop, []))
        intersect = sorted(set(desired_values) & set(frag_values))
        if intersect:
            matching_fragments.append(fragment)

print(f"Found {len(matching_fragments)} fragments matching at least one provided property-value pair")

# Display each fragment
for i, fragment in enumerate(matching_fragments, 1):
    print()
    print(f"Fragment {i}: {fragment.to_json()}")

Found 1 fragments matching at least one provided property-value pair

Fragment 1: {"entity_id": "Q100067", "properties": {"name": ["Nawada district"], "country": ["India"]}, "source_ids": [], "evidence_map": {}, "metadata": {"fragment_id": "4225724", "title": "Kutri", "title_entity_id": "Q86751968", "type": ["district of India"]}}
